In [3]:
import numpy as np
import matplotlib.pyplot as plt 

%matplotlib

Using matplotlib backend: module://matplotlib_inline.backend_inline


In [1]:
import math


# Gasket
gasket_ID, gasket_OD = 92.8, 112.8

N = (gasket_OD - gasket_ID)/2
bo = N/2                                                                           # basic gasket seating width
b = bo if bo <= 6 else 0.5*math.sqrt(bo)                                           # b0<=6mm -> b=b0
G = (gasket_OD + gasket_ID)/2 if bo <= 6 else gasket_OD - 2*b                      # mean dia
P = 2.0                                                                            # MPa, operating pressure

m = 2.5                                                                            # min. compressive force for sealing, gasket factor, APX2 HOCHDRUCK V15011W3
y_psi = 3000                                                                       # gasket seating stress, APX2 HOCHDRUCK V15011W3
psi = 0.0068929                                                                    # conversion factor: psi to MPa 
y = y_psi*psi                                                                      # gasket unit seating load

# --- Bolt loads, ASME App 2 ---
H = 0.785*(G**2)*P                                                                 # hydrostatic end force, APX2 
Hp = 2*b*math.pi*G*m*P                                                             # extra bolt load required to seat gasket under pressure from H
Wm1 = H + Hp                                                                       # minimum required bolt load for opearting condition
Wm2 = math.pi*G*b*y                                                                # minimum required bolt load for gasket seating condition
Wgov = max(Wm1, Wm2)                                                               # governing
gmode = "seating (Wm2)" if Wm2 >= Wm1 else "operating (Wm1)"                       # which load governs
#----------------------------------------------------------------------------------------------------------------------------\

# Bolts
n_bolt = 8                                                                          # number of bolts
As_bolt = 36.6                                                                      # mm^2, bolt stress area
    

# --- Gasket seating-stress check (delivered over effective area = pi*G*b) ---
A_eff = math.pi*G*b                                                                 # gasket effective area seating
sigma_seat = Wm2/A_eff                                                              # applied gasket seating stress, MPa
VU = 20                                                                             # unterer Wert; minimum allowable stress to seat, N/mm^2
VO = 270                                                                            # max allowable stress at 20 deg. C, N/mm^2
BO300 = 210                                                                         # max allowable stress at 300 deg. C, N/mm^2



# interpolationn of yld_hot using mechanical properties of A286 from source: https://alloya286.com/mechanical-properties

M1 = 527
U1 = 540
L1 = 425

U2 = 580
L2 = 595

yld_hot = ((((M1-U1)*(L2-U2)))/(L1-U1))+U2                                           # A286 datasheet hot yield at 527C, MPa (governing basis)


# --- Bolt margins: 8 x M8 A286 ---
Ab = n_bolt*As_bolt                                                                 # 292.8 ~ 293 mm^2
sigma_bolt_gov = Wgov/Ab                                                            # applied bolt stress at governing load, MPa
preload = 1.4*Wm1                                                                   # assembly preload target (tightness), N
sigma_preload = preload/Ab


In [14]:
print(f"geometry:  N={N:.1f}  bo={bo:.1f}  b={b:.1f} mm  G={G:.1f} mm  (ASME Table 2-5.2)")
print(f"factors :  m={m}  y={y_psi:.0f} psi = {y:.2f} N/mm^2      [APX2 HOCHDRUCK V15011W3, ASTM]")
print("-"*60)
print(f"H   = {H/1000:6.2f} kN   Hp = {Hp/1000:6.2f} kN")
print(f"Wm1 = {Wm1/1000:6.2f} kN  (operating)")
print(f"Wm2 = {Wm2/1000:6.2f} kN  (seating)")
print(f"Wgov= {Wgov/1000:6.2f} kN  -> GOVERNING = {gmode}")
print("-"*60)
print(f"gasket seating stress (over pi*G*b={A_eff:.0f} mm^2) = {sigma_seat:.1f} N/mm^2")
print(f"   check: VU {VU} <= {sigma_seat:.1f} <= VO {VO} (20 C) / BO {BO300} (300 C)  -> seated, not crushed")
print("-"*60)
print(f"bolts: {n_bolt} x M8 A286, Ab = {Ab:.0f} mm^2")
print(f"Interpolated hot yield strength of A286 at 527 C: {yld_hot:.2f} MPa")
print(f"applied bolt stress @Wgov = {sigma_bolt_gov:.0f} MPa = {sigma_bolt_gov/yld_hot*100:.0f}% of 582 hot yield")
print(f"preload 1.4xWm1 = {preload/1000:.1f} kN total = {sigma_preload:.0f} MPa = {sigma_preload/yld_hot*100:.0f}% hot yield")
print(f"envelope check: Wgov {Wgov/1000:.1f} kN <= 35 kN design envelope: {'OK' if Wgov<=35000 else 'EXCEEDS'}")

geometry:  N=10.0  bo=5.0  b=5.0 mm  G=102.8 mm  (ASME Table 2-5.2)
factors :  m=2.5  y=3000 psi = 20.68 N/mm^2      [APX2 HOCHDRUCK V15011W3, ASTM]
------------------------------------------------------------
H   =  16.59 kN   Hp =  16.15 kN
Wm1 =  32.74 kN  (operating)
Wm2 =  33.39 kN  (seating)
Wgov=  33.39 kN  -> GOVERNING = seating (Wm2)
------------------------------------------------------------
gasket seating stress (over pi*G*b=1615 mm^2) = 20.7 N/mm^2
   check: VU 20 <= 20.7 <= VO 270 (20 C) / BO 210 (300 C)  -> seated, not crushed
------------------------------------------------------------
bolts: 8 x M8 A286, Ab = 293 mm^2
Interpolated hot yield strength of A286 at 527 C: 581.70 MPa
applied bolt stress @Wgov = 114 MPa = 20% of 582 hot yield
preload 1.4xWm1 = 45.8 kN total = 157 MPa = 27% hot yield
envelope check: Wgov 33.4 kN <= 35 kN design envelope: OK
